# Dataspace Contract Negotiation & PUSH Transfer Flow

This notebook demonstrates an end-to-end Dataspace Protocol (DSP 2025-1) lifecycle in PUSH mode:
1. Executes prerequisite setup (`setup_catalog_policies.ipynb`) to initialize participants, catalog, policies, and connectors.
2. Performs full DSP 2025-1 Contract Negotiation between Consumer and Provider (Request -> Offer -> Counter-Request -> Acceptance -> Agreement -> Verification -> Finalization).
3. Executes DSP PUSH Transfer Process using Eunomia RPC (`/dsp/current/transfers/rpc/setup-*`).
4. Exercises transfer lifecycle state transitions: `REQUESTED` -> `STARTED` -> `SUSPENDED` -> `STARTED` (resumed) -> `COMPLETED`.
5. Validates DSP protocol role enforcement rules (rejecting Consumer-initiated start).
6. Completes the transfer and queries final state via the Transfer Agent Management API.

## 1. Prerequisites
Execute the catalog and policies notebook to establish onboarding, catalog entries, policies, and connectors.

In [4]:
%run ./setup_catalog_policies.ipynb

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500)
Wallet linked for Consumer (http://127.0.0.1:1100)
Wallet linked for Provider (http://127.0.0.1:1200)
Authority DID: did:web:127.0.0.1%3A1500

Consumer DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ

Provider DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dE

# Part I: Contract Negotiation (DSP 2025-1)
Complete negotiation sequence between Consumer and Provider following DSP 2025-1 state transitions.

## 2. Negotiation Request Initiation (Consumer -> Provider)
The Consumer initiates contract negotiation by sending a `ContractRequestMessage` via `/dsp/current/negotiations/rpc/setup-request-init` referencing the target dataset and offer policy.
The Provider assigns a `providerPid` and transitions the negotiation state to `REQUESTED`.

In [5]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:c7f54e6c-6a95-44c2-ba56-c9e337ba37a0",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
  

## 3. Provider Initial Offer (Provider -> Consumer)
The Provider responds with a counter-offer by sending a `ContractOfferMessage` via `/dsp/current/negotiations/rpc/setup-offer`.
The negotiation transitions to `OFFERED`.

In [6]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:c7f54e6c-6a95-44c2-ba56-c9e337ba37a0",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:d5a2e897-5bab-4ce6-9d32-a5ff2fd0ffa9"
    },
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2",
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:bd11553f-f6a7-4e6c-85

## 4. Consumer Counter-Request Based on Offer (Consumer -> Provider)
The Consumer reviews the offer and submits a counter-request via `/dsp/current/negotiations/rpc/setup-request`.
The negotiation state transitions back to `REQUESTED`.

In [7]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:c7f54e6c-6a95-44c2-ba56-c9e337ba37a0",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:d5a2e897-5bab-4ce6-9d32-a5ff2fd0ffa9"
    },
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2",
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:d2855773-c736-4360-bc80-39eeeb7cf1ad",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgen

## 5. Provider Offer Confirmation (Provider -> Consumer)
The Provider confirms or updates the terms of the offer via `/dsp/current/negotiations/rpc/setup-offer`.
The negotiation state transitions to `OFFERED`.

In [8]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:c7f54e6c-6a95-44c2-ba56-c9e337ba37a0",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:d5a2e897-5bab-4ce6-9d32-a5ff2fd0ffa9"
    },
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2",
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:bd11553f-f6a7-4e6c-855a-d1507ba217ed",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "OFFERED",
    "stateAttribute": null,
    "associated

## 6. Consumer Offer Acceptance (Consumer -> Provider)
The Consumer accepts the Provider's offer terms via `/dsp/current/negotiations/rpc/setup-acceptance`.
The negotiation state transitions to `ACCEPTED`.

In [9]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2",
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:d2855773-c736-4360-bc80-39eeeb7cf1ad",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdH

## 7. Provider Contract Agreement (Provider -> Consumer)
The Provider generates the binding `ContractAgreementMessage` containing the finalized agreement terms and sends it to the Consumer via `/dsp/current/negotiations/rpc/setup-agreement`.
The negotiation state transitions to `AGREED`.

In [10]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2",
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:bd11553f-f6a7-4e6c-855a-d1507ba217ed",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQ

## 8. Consumer Agreement Verification (Consumer -> Provider)
The Consumer verifies receipt and terms of the contract agreement by sending a `ContractAgreementVerificationMessage` via `/dsp/current/negotiations/rpc/setup-verification`.
The negotiation state transitions to `VERIFIED`.

In [11]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2",
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:d2855773-c736-4360-bc80-39eeeb7cf1ad",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdH

## 9. Provider Negotiation Finalization (Provider -> Consumer)
The Provider acknowledges verification and sends a `ContractNegotiationEventMessage` (finalization) via `/dsp/current/negotiations/rpc/setup-finalization`.
The negotiation transitions to the terminal state `FINALIZED`.

In [12]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:25d7974c-d5e3-4d5a-863b-59b6436882f2",
    "providerPid": "urn:provider-pid:6506adad-9de0-4490-8d7b-3a52eb6207a2",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:bd11553f-f6a7-4e6c-855a-d1507ba217ed",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd

## 10. Final Agreement Details
Display the finalized contract agreement and extract the `agreement_id` required for subsequent transfer requests.

In [13]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:1b0a10dd-3bf2-42db-8c45-7481d7fdb074",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Public",
  "negotiationAgentProcessId": "urn:negotiation-process:bd11553f-f6a7-4e6c-855a-d1507ba217ed",
  "negotiationAgentMessageId": "urn:negotiation-message:ccd78def-b9fd-41e7-a475-80f22f86bd57",
  "consumerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "providerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXN

# Part II: Transfer Process Execution (PUSH Mode via Eunomia RPC)

In **PUSH mode**, the Consumer specifies a destination data sink (`dataAddress`) in the initial transfer request.
The Provider connects to the designated sink and pushes data directly to it upon start.
Both peers interact with their local transfer agents through Eunomia RPC (`/dsp/current/transfers/rpc/setup-*`).

## 11. Transfer Request Setup with Sink DataAddress (Consumer -> Provider)
The Consumer initiates the PUSH transfer by calling `/dsp/current/transfers/rpc/setup-request`, specifying the format `http+pushing` and providing its destination sink `dataAddress` where data should be delivered.
The Provider accepts the request, creates the process in `REQUESTED` state, and returns `providerPid`.

In [14]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+pushing",
    "dataAddress": {
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://127.0.0.1:1112",
        "endpointProperties": [
            {
                "name": "authorization",
                "value": "TOKEN-ABCDEFG"
            },
            {
                "name": "authType",
                "value": "bearer"
            }
        ]
    },
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["transferAgentModel"]["correlation"]["consumerPid"]
transfer_process_provider_pid = response_as_json["transferAgentModel"]["correlation"]["providerPid"]
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "agreementId": "urn:agreement:1b0a10dd-3bf2-42db-8c45-7481d7fdb074",
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "dataAddress": {
      "endpoint": "http://127.0.0.1:1112",
      "endpointProperties": [
        {
          "na

## 12. Start Push Transfer (Provider -> Consumer)
The Provider invokes `/dsp/current/transfers/rpc/setup-start`. The Provider's dataplane begins pushing data to the Consumer's designated sink.
The transfer process moves from `REQUESTED` to `STARTED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 13. Suspend Transfer (Provider -> Consumer)
The Provider temporarily suspends the active data transmission by calling `/dsp/current/transfers/rpc/setup-suspension`.
The process transitions from `STARTED` to `SUSPENDED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 14. Resume Push Transfer (Provider -> Consumer)
The Provider resumes the suspended push transfer by sending a `RpcTransferStartMessage` to `/dsp/current/transfers/rpc/setup-start`.
The transfer process returns to `STARTED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 15. Role Validation Test: Reject Consumer-Initiated Start
Verify DSP role constraint enforcement: In DSP 2025-1, only a Provider may send a `TransferStartMessage`.
Attempting to invoke `/dsp/current/transfers/rpc/setup-start` as the Consumer fails with a role validation error.

In [ ]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
print("Status Code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)


## 16. Complete Transfer (Provider -> Consumer)
Once the Provider has delivered all data items to the Consumer's sink, the Provider signals completion via `/dsp/current/transfers/rpc/setup-completion`.
A `TransferCompletionMessage` is sent over DSP, moving the transfer process to the terminal state `COMPLETED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferCompletionMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 17. Query Final Transfer Process State
Query the Provider's Transfer Agent Management API (`/api/v1/transfer-agent/transfer-processes`) to inspect the final recorded process state.

In [ ]:
url = data_space_provider + "/api/v1/transfer-agent/transfer-processes"
response = requests.request("GET", url, headers=json_header)
print("Processes in Provider:")
print(json.dumps(response.json(), indent=2))
